# Fast approximation of a cylinder magnet's B-field

**Goal:** a magnet is a 6mm-diameter x 6mm-tall cylinder, magnetized along its
axis. We want a *fast* (non-magpylib-call) approximation of its B-field over
a rectangular region below it, good enough for quick repeated evaluation.

**Region of interest:** originally a 10x10x10mm cube with its top face
centered on the magnet's bottom face. We found that the field very close to
the magnet's circular edge (radius 3mm, right at the top face) is a real
physical singularity that breaks a single smooth polynomial fit. So here we
**crop the top 2mm off the box**, leaving a 10 (x) x 10 (y) x 8 (z) mm region
that starts 2mm below the magnet face. This keeps us clear of the edge
singularity and (as shown below) makes the fit dramatically more accurate.

**Approach:** since the magnet is axisymmetric, the field has no
phi-dependence and no B_phi component. We exploit this by fitting an
**axisymmetric harmonic polynomial** (regular solid harmonics in cylindrical
coordinates) rather than a generic 3D Taylor series -- fewer coefficients,
and it automatically satisfies Maxwell's equations (div B = 0, curl B = 0)
in the source-free region.


In [1]:
import time
import numpy as np
import scipy
import sympy as sp
import matplotlib.pyplot as plt
import magpylib as magpy


In [ ]:
a = np.random.random((3, 3)).astype(np.float32) * 500
def hex_array_2d(arr):
    lines = []
    for row in arr:
        row_string = ", ".join(
            x.hex() +"f" for x in row.tolist()
        )
        lines.append(f"Vec3{{{row_string}}}")
    return "{\n" + ",\n".join(lines) + "\n}"

print(hex_array_2d(a))

In [ ]:
a = np.random.random((3, 3)).astype(np.float32) * 500

string = "{\n"
for row in a:
    string += "{"
    for x in row:
        string += float(x).hex() + "f"
    string += "},\n"
string += "}"
print(string)

## 1. Define the magnet and the region of interest

Units throughout: millimeters for length, millitesla (mT) for field.

**Unit note (magpylib v5):** use `polarization` (mT, = remanence field Br) for
magnet strength, not `magnetization` -- that kwarg expects A/m and silently
gives a field ~1000x too weak if handed mT.

**Where the magnet actually lives:** the magnet definition and
`BICUBIC_FIELD_REFERENCE_MT` are in `bicubic_table.py`, not this cell --
both this notebook and `generate_bicubic_table.py` (the script that
actually writes the firmware files) import them from there, so the two
can never describe a different magnet.


In [ ]:
# --- Magnet: 6mm diameter x 6mm height cylinder, magnetized along +z ---
# See bicubic_table.py for the magnet definition and what
# BICUBIC_FIELD_REFERENCE_MT means (short version: it's the polarization the
# table is generated at, arbitrary since field scales linearly with it).
from bicubic_table import BICUBIC_FIELD_REFERENCE_MT, build_magnet

magnet = build_magnet()

# --- Region of interest: 10x10x8mm box, cropped 2mm off the top ---
# Original cube: x,y in [-5,5], z in [-10,0] (top face touching magnet's bottom face)
# Cropped box:   x,y in [-5,5], z in [-10,-2]  (removes the 2mm layer nearest the magnet)
X_MIN, X_MAX = -5.0, 5.0
Y_MIN, Y_MAX = -5.0, 5.0
Z_TOP, Z_BOT = -2.0, -10.0   # top face now 2mm below the magnet, not touching it

Z_CENTER = (Z_TOP + Z_BOT) / 2.0   # -6.0mm; on-axis, so this is our expansion point
print(f"Box: x,y in [{X_MIN},{X_MAX}], z in [{Z_BOT},{Z_TOP}]  (center at z={Z_CENTER})")


## 2. Compute & visualize the true field (magpylib) over the box

Quick sanity-check plots so we can see what we're approximating.


In [ ]:
# Cross-section (x-z plane at y=0), covering a bit above the box top too
nx, nz = 1200, 1200
x_line = np.linspace(-5, 5, nx)
z_line = np.linspace(-10, 2, nz)   # small margin above to show the magnet & cropped-out layer

Xs, Zs = np.meshgrid(x_line, z_line)
pts = np.stack([Xs.ravel(), np.zeros_like(Xs.ravel()), Zs.ravel()], axis=-1)

B = magnet.getB(pts).reshape(nx, nz, 3)

In [ ]:

Bx, Bz = B[..., 0], B[..., 2]
Bmag = np.linalg.norm(B, axis=-1)

fig, ax = plt.subplots(figsize=(6, 6))
pcm = ax.pcolormesh(Xs, Zs, Bmag, shading="auto", cmap="magma",
                     norm=plt.matplotlib.colors.LogNorm(vmin=max(Bmag.min(), 1e-2), vmax=Bmag.max()/2))
fig.colorbar(pcm, ax=ax, label="|B| (mT, log scale)")

ax.streamplot(Xs, Zs, Bx, Bz, color='white', density=1.5, linewidth=0.5, arrowsize=1.0)


ax.add_patch(plt.Rectangle((-3, 0), 6, 6, fill=False, edgecolor="cyan", linewidth=2, label="magnet"))
ax.add_patch(plt.Rectangle((-5, Z_BOT), 10, Z_TOP - Z_BOT, fill=False, edgecolor="lime", linewidth=2, label="cropped box"))
ax.add_patch(plt.Rectangle((-5, Z_TOP), 10, -Z_TOP, fill=False, edgecolor="red", linestyle="--", linewidth=1.5, label="removed 2mm layer"))

ax.set_xlabel("x (mm)"); ax.set_ylabel("z (mm)")
ax.set_title("B-field magnitude & direction (y=0 cross-section)")
ax.legend(loc="upper right", fontsize=8)
ax.set_aspect("equal")
fig.tight_layout()
plt.show()


In [ ]:
np.min(Bmag)

## 2b. How much does the finite cylinder shape actually matter?

Before building a bicubic table, it's worth checking whether we even need a
shape-aware model: does a plain point dipole already approximate the field
well enough over the ROI? magpylib guarantees that any homogeneously
magnetized body's far field converges to that of a point dipole with
`moment = magnetization * volume` (available directly as `magnet.dipole_moment`).
We compare that dipole's field against the true cylinder field over the same
cropped box used above.


In [ ]:
# Equivalent point dipole: same position, moment = magnetization * volume
dipole = magpy.misc.Dipole(moment=magnet.dipole_moment, position=magnet.position)
print(f"Cylinder volume: {magnet.volume:.2f} mm^3")
print(f"Dipole moment:   {magnet.dipole_moment} mT*mm^3")

nx_cmp, nz_cmp = 300, 300
x_cmp = np.linspace(X_MIN, X_MAX, nx_cmp)
z_cmp = np.linspace(Z_BOT, Z_TOP, nz_cmp)
Xc, Zc = np.meshgrid(x_cmp, z_cmp)
pts_cmp = np.stack([Xc.ravel(), np.zeros_like(Xc.ravel()), Zc.ravel()], axis=-1)

B_cyl_cmp = magnet.getB(pts_cmp).reshape(nx_cmp, nz_cmp, 3)
B_dip_cmp = dipole.getB(pts_cmp).reshape(nx_cmp, nz_cmp, 3)

dip_err = B_dip_cmp - B_cyl_cmp
dip_err_mag = np.linalg.norm(dip_err, axis=-1)
dip_err_rel = 100 * dip_err_mag / np.linalg.norm(B_cyl_cmp, axis=-1)

print(f"Relative error over ROI: mean={dip_err_rel.mean():.1f}%, "
      f"median={np.median(dip_err_rel):.1f}%, max={dip_err_rel.max():.1f}%")


In [ ]:
err_x, err_z = dip_err[..., 0], dip_err[..., 2]   # direction of the dipole-minus-cylinder discrepancy

fig, ax = plt.subplots(2, 2, figsize=(12, 12))

pcm = ax[0, 0].pcolormesh(Xc, Zc, dip_err_mag, shading="auto", cmap="magma",
                           norm=plt.matplotlib.colors.LogNorm(vmin=max(dip_err_mag.min(), 1e-3), vmax=dip_err_mag.max()))
ax[0, 0].streamplot(Xc, Zc, err_x, err_z, color='white', density=1.2, linewidth=0.5, arrowsize=1.0)
ax[0, 0].set_title("|B_dipole - B_cylinder| (mT), with error field lines")
ax[0, 0].set_xlabel("x (mm)"); ax[0, 0].set_ylabel("z (mm)")
fig.colorbar(pcm, ax=ax[0, 0])

ax[0, 1].hist(dip_err_mag.ravel(), bins=100, log=True)
ax[0, 1].set_title("Absolute error histogram")

pcm = ax[1, 0].pcolormesh(Xc, Zc, dip_err_rel, shading="auto", cmap="magma",
                           norm=plt.matplotlib.colors.LogNorm(vmin=max(dip_err_rel.min(), 1e-2), vmax=dip_err_rel.max()))
ax[1, 0].streamplot(Xc, Zc, err_x, err_z, color='white', density=1.2, linewidth=0.5, arrowsize=1.0)
ax[1, 0].set_title("Relative error (%) over ROI, with error field lines")
ax[1, 0].set_xlabel("x (mm)"); ax[1, 0].set_ylabel("z (mm)")
fig.colorbar(pcm, ax=ax[1, 0])

ax[1, 1].hist(dip_err_rel.ravel(), bins=100, log=True)
ax[1, 1].set_title("Relative error histogram (%)")

fig.tight_layout()
plt.show()


In [ ]:
# On-axis (r=0) comparison, and where the dipole approximation converges to truth
z_axis = np.linspace(Z_BOT, -0.1, 500)
pts_axis = np.stack([np.zeros_like(z_axis), np.zeros_like(z_axis), z_axis], axis=-1)
B_cyl_axis = np.linalg.norm(magnet.getB(pts_axis), axis=-1)
B_dip_axis = np.linalg.norm(dipole.getB(pts_axis), axis=-1)
axis_err_rel = 100 * np.abs(B_dip_axis - B_cyl_axis) / B_cyl_axis

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))

ax[0].plot(-z_axis, B_cyl_axis, label="cylinder (truth)")
ax[0].plot(-z_axis, B_dip_axis, label="point dipole", linestyle="--")
ax[0].axvspan(-Z_TOP, -Z_BOT, color="lime", alpha=0.15, label="ROI (on-axis)")
ax[0].set_xlabel("distance below magnet center, -z (mm)")
ax[0].set_ylabel("|B| (mT)")
ax[0].set_yscale("log")
ax[0].legend()
ax[0].set_title("On-axis field: cylinder vs. equivalent dipole")

ax[1].plot(-z_axis, axis_err_rel)
ax[1].axvspan(-Z_TOP, -Z_BOT, color="lime", alpha=0.15, label="ROI (on-axis)")
ax[1].set_xlabel("distance below magnet center, -z (mm)")
ax[1].set_ylabel("relative error (%)")
ax[1].set_title("On-axis dipole approximation error")
ax[1].legend()

fig.tight_layout()
plt.show()


## 3. Build the bicubic interpolation table

This is the actual table `generate_bicubic_table.py` writes to the
firmware -- computed here from the same `bicubic_table.py` functions the
script uses, purely so the cells below can inspect/plot it. **This
notebook does not write the firmware files any more** -- run
`python generate_bicubic_table.py` from a terminal to do that.


In [ ]:
from bicubic_table import compute_field_table

table = compute_field_table(magnet)
x_interp_line, z_interp_line = table.r_line, table.z_line
B_for_pts = table.field

B_interpolator = scipy.interpolate.RegularGridInterpolator((x_interp_line, z_interp_line), B_for_pts, method='cubic', bounds_error=True)

# Xs_interp/Zs_interp: grid-point overlay for the error plot below --
# reconstructed here (not part of bicubic_table.py) since it's a
# notebook-only inspection aid, not something the firmware needs.
Xs_interp, Zs_interp = np.meshgrid(x_interp_line, z_interp_line, indexing='ij')


In [ ]:
test_x_line = np.linspace(x_interp_line[0], x_interp_line[-1], 1000)
test_z_line = np.linspace(z_interp_line[0], z_interp_line[-1], 1000)    
test_X, test_Y = np.meshgrid(test_x_line, test_z_line, indexing='ij')

B_interp = B_interpolator((test_X, test_Y)).reshape(1000, 1000, 3)
B_truth  = magnet.getB(np.stack([test_X.ravel(), np.zeros_like(test_X.ravel()), test_Y.ravel()], axis=-1)).reshape(1000, 1000, 3)

interp_error = B_interp - B_truth
error_mag = np.linalg.norm(interp_error, axis=-1)

interp_error_rel = 100 % np.divide(np.linalg.norm(interp_error, axis=-1), np.linalg.norm(B_truth, axis=-1), out=np.zeros_like(error_mag), where=np.linalg.norm(B_truth, axis=-1)!=0)



fig, ax = plt.subplots(2, 2, figsize=(12, 12))
pcm = ax[0, 0].pcolormesh(test_X, test_Y, error_mag, shading="auto", cmap="magma",
                        norm=plt.matplotlib.colors.LogNorm(vmin=error_mag.min() + 0.001 * error_mag.max(), vmax=error_mag.max()))
ax[0, 0].scatter(Xs_interp, Zs_interp, s=1, color='white', marker='+', alpha=0.5)
ax[0, 0].set_title("Interpolation Error")
ax[0, 0].set_xlabel("X (mm)")
ax[0, 0].set_ylabel("Z (mm)")
ax[0, 1].hist(error_mag.ravel(), bins=100, log=True)
fig.colorbar(pcm, label="|B| (mT, log scale)")

pcm = ax[1, 0].pcolormesh(test_X, test_Y, interp_error_rel, shading="auto", cmap="magma",
                          norm=plt.matplotlib.colors.LogNorm(vmin=interp_error_rel.min() + 0.001 * interp_error_rel.max(), vmax=interp_error_rel.max()))
ax[1, 0].scatter(Xs_interp, Zs_interp, s=1, color='white', marker='+', alpha=0.5)
ax[1, 0].set_title("relative Interpolation Error in %")
ax[1, 0].set_xlabel("X (mm)")
ax[1, 0].set_ylabel("Z (mm)")
ax[1, 1].hist(interp_error_rel.ravel(), bins=100, log=True)

fig.colorbar(pcm, label="|B| (mT, log scale)")

plt.show()

In [ ]:
# Preview only -- this no longer writes the firmware files. The real
# generation entry point is `python generate_bicubic_table.py`, which calls
# these exact same functions (format_header/format_cpp) on the exact same
# `table` computed above.
from bicubic_table import format_header, format_cpp

header = format_header(table, BICUBIC_FIELD_REFERENCE_MT)
cpp_file = format_cpp(table)


In [ ]:
print(header)

print(cpp_file)